# 音乐源分离 - pymss WebUI / ComfyUI

This notebook installs [pymss](https://github.com/pymss-project/pymss) (Python Music Source Separation) on Google Colab with GPU, and supports both **pymss WebUI** and **[ComfyUI](https://github.com/comfyanonymous/ComfyUI)** + [comfy-mss](https://github.com/pymss-project/comfy-mss) nodes.

**What you get:** A browser-accessible interface where you can upload audio and separate vocals, drums, bass, and other instruments.

**Requirements:** Colab runtime with GPU (Runtime > Change runtime type > T4 GPU).

## 1. Install dependencies

In [ ]:
#@title 1. Install dependencies { display-mode: "form" }
# ============================================================
# pymss Colab - Step 1: Install
# ============================================================

import torch, subprocess, shutil, os

# ---- Check GPU ----
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Runtime > Change runtime type > GPU.")
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"[GPU] {gpu_name} ({gpu_mem:.1f} GB)")

# ---- Install pymss[server] ----
!pip install -q "pymss[server]"
pymss_ver = subprocess.check_output(["pip", "show", "pymss"], text=True)
pymss_ver = [l.split(":", 1)[1].strip() for l in pymss_ver.splitlines() if l.startswith("Version:")][0]
print(f"[pymss] {pymss_ver}")

# ---- Download cloudflared ----
if not shutil.which("cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared
cf_ver = subprocess.check_output(["cloudflared", "--version"], text=True).strip()
print(f"[cloudflared] {cf_ver}")

# ---- Install ComfyUI + comfy-mss (optional, for comfyui mode) ----
COMFYUI_DIR = "/content/ComfyUI"
if not os.path.isdir(COMFYUI_DIR):
    !git clone https://github.com/comfyanonymous/ComfyUI.git {COMFYUI_DIR}
!pip install -q -r {COMFYUI_DIR}/requirements.txt
# ComfyUI Manager
comfy_manager_dir = os.path.join(COMFYUI_DIR, "custom_nodes", "ComfyUI-Manager")
if not os.path.isdir(comfy_manager_dir):
    !git clone https://github.com/Comfy-Org/ComfyUI-Manager.git {comfy_manager_dir}
!pip install -q -r {comfy_manager_dir}/requirements.txt
# comfy-mss custom node
comfy_mss_dir = os.path.join(COMFYUI_DIR, "custom_nodes", "comfy-mss")
if not os.path.isdir(comfy_mss_dir):
    !git clone https://github.com/pymss-project/comfy-mss.git {comfy_mss_dir}
print(f"[comfyui] ComfyUI + comfy-mss installed at {COMFYUI_DIR}")

print("\nInstallation complete. Run the next cell to start the server.")

## 2. Start server

In [ ]:
#@title 2. Start server { display-mode: "form" }
# ============================================================
# pymss Colab - Step 2: Start server
# ============================================================

import subprocess, time, urllib.request, sys, threading, os

# ---- Select mode ----
MODE = "webui" #@param ["webui", "comfyui"]

PORT = 8188 #@param {type:"integer"}
SERVER_LOG = "/tmp/pymss_server.log"

def stream_server_log(proc):
    """Read server log file and print new lines to cell output."""
    last_pos = 0
    while proc.poll() is None:
        try:
            with open(SERVER_LOG, "r") as f:
                f.seek(last_pos)
                new = f.read()
                if new:
                    print(new, end="", flush=True)
                    last_pos = f.tell()
        except FileNotFoundError:
            pass
        time.sleep(0.5)

if MODE == "comfyui":
    # ---- Start ComfyUI ----
    print("[comfyui] Starting ComfyUI server...")
    log_file = open(SERVER_LOG, "w")
    server_proc = subprocess.Popen(
        [sys.executable, "main.py",
         "--listen", "0.0.0.0", "--port", str(PORT),
         "--enable-manager"],
        cwd=COMFYUI_DIR,
        stdout=log_file, stderr=subprocess.STDOUT,
    )
    log_thread = threading.Thread(target=stream_server_log, args=(server_proc,), daemon=True)
    log_thread.start()
    for _ in range(180):
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{PORT}/", timeout=2)
            print(f"[comfyui] Server ready on port {PORT}.")
            break
        except Exception:
            time.sleep(1)
    else:
        raise RuntimeError("ComfyUI failed to start within 180s.")
else:
    # ---- Start pymss WebUI ----
    print("[pymss] Starting WebUI server (model loads on first request)...")
    log_file = open(SERVER_LOG, "w")
    server_proc = subprocess.Popen(
        ["pymss", "serve", "--debug",
         "--host", "0.0.0.0", "--port", str(PORT),
         "--device", "auto", "--source", "modelscope", "--webui"],
        stdout=log_file, stderr=subprocess.STDOUT,
    )
    log_thread = threading.Thread(target=stream_server_log, args=(server_proc,), daemon=True)
    log_thread.start()
    for _ in range(120):
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=2)
            print(f"[pymss] Server ready on port {PORT}.")
            break
        except Exception:
            time.sleep(1)
    else:
        raise RuntimeError("pymss server failed to start within 120s.")

print("Server is running. Run the next cell to expose a public URL.")

## 3. Start public tunnel

In [ ]:
#@title 3. Start public tunnel { display-mode: "form" }
# ============================================================
# pymss Colab - Step 3: Launch public tunnel
# ============================================================

import re, tempfile, os

tunnel_log = tempfile.NamedTemporaryFile(mode="w", suffix=".log", delete=False)
print("[tunnel] Starting cloudflared...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=tunnel_log, stderr=subprocess.STDOUT,
)
url_re = re.compile(r"(https://[a-z0-9\-]+\.trycloudflare\.com)")
public_url = None
start = time.time()
while time.time() - start < 30:
    try:
        with open(tunnel_log.name) as f:
            content = f.read()
        m = url_re.search(content)
        if m:
            public_url = m.group(1)
            break
    except FileNotFoundError:
        pass
    time.sleep(0.5)
os.unlink(tunnel_log.name)
if not public_url:
    raise RuntimeError("Tunnel failed to start within 30s.")

print(f"\n{'='*60}")
print(f"  Mode:        {MODE}")
print(f"  Public URL:  {public_url}")
if MODE == "comfyui":
    print(f"  WebUI:       {public_url}/")
else:
    print(f"  WebUI:       {public_url}/ui/")
    print(f"  API docs:    {public_url}/docs")
print(f"{'='*60}\n")
print("Open the WebUI link above to start separating audio.")
print("Stop: tunnel_proc.terminate() + server_proc.terminate()")

---

### Quick Reference

**Mode switch:** Change `MODE = "webui"` to `MODE = "comfyui"` in Step 2, then re-run Step 2 & 3.

**WebUI (pymss):** Open URL → browse catalog → load model → upload audio → separate.

**ComfyUI:** Open URL → build/load workflow with MSS Separate nodes → queue prompt.

**Python API:**
```python
from pymss import MSSeparator
with MSSeparator.from_model_name("bs_roformer_voc_hyperacev2", download=True, device="auto", output_format="wav", store_dirs="results") as sep:
    sep.process_folder("/path/to/audio")
```

**Cleanup:**
```python
tunnel_proc.terminate()
server_proc.terminate()
```